## The Core Abstraction: From Problem to Hardware

When you write high-performance GPU code, you are usually solving a very large problem, such as applying an operation to one million array elements.

The central question is how to split that work so thousands of physical ALUs can run it in parallel without interfering with each other.

CUDA and Triton organize work in a hierarchy:

| Logical abstraction | Hardware mapping |
| --- | --- |
| Grid | Entire GPU die |
| Thread block | One SM |
| Warp | 32 execution lanes |
| Thread | One logical lane of execution |

```text
GRID ──► Entire GPU Die
BLOCK ──► Assigned to one SM
WARP ──► 32 physical ALUs / SIMT lanes
THREAD ──► One logical execution lane
```

Let us unpack each layer from top to bottom.

## 2. The Hierarchy Dissected

### A. The Thread

A thread is the smallest logical unit of execution. It has:

- a unique ID such as threadIdx.x,
- private registers,
- private local state.

In practice, the hardware does not schedule a single thread in isolation. It groups threads into warps of 32.

### B. The Warp

A warp is a group of 32 consecutive threads that execute the same instruction together.

This is the fundamental execution unit for SIMT:

- one instruction is issued,
- all 32 lanes execute it on different data,
- they run in lockstep.

### C. The Thread Block / Cooperative Thread Array (CTA)

A thread block is a user-defined group of threads, often 128, 256, 512, or 1024 threads.

A key guarantee is that all threads in one block execute on the same physical SM.

This makes block-level cooperation efficient because threads can use:

- shared memory,
- low-latency communication,
- barriers such as __syncthreads() or tl.sync().

Threads in different blocks do not share that guarantee.

### D. The Grid

A grid is the full collection of all thread blocks launched for a kernel. The hardware distributes the grid across the available SMs on the GPU.

---

## 3. How a Thread Block Is Sliced into Warps

If a thread block has 128 threads, the hardware splits it into four warps:

- Warp 0: threads 0–31
- Warp 1: threads 32–63
- Warp 2: threads 64–95
- Warp 3: threads 96–127

```text
Thread Block (128 Threads)
┌─────────────────────────────────────────────────────────┐
│ Warp 0: [T0,  T1,  T2,  ...  T31]  ──► Scheduler       │
│ Warp 1: [T32, T33, T34, ...  T63]  ──► Scheduler       │
│ Warp 2: [T64, T65, T66, ...  T95]  ──► Scheduler       │
│ Warp 3: [T96, T97, T98, ... T127]  ──► Scheduler       │
└─────────────────────────────────────────────────────────┘
```

A practical rule is to make block sizes multiples of 32. If a block has 40 threads, the hardware still allocates two full warps, and the last 24 lanes are effectively idle.

---

## 4. SIMT Execution Mechanics and Warp Scheduling

An SM may hold many warps at once. The scheduler hides memory latency by switching between ready warps while other warps wait for data.

```text
Cycle 0: Warp 0 issues a load from HBM
Cycle 1: Scheduler switches to Warp 1
Cycle 2: Scheduler switches to Warp 2
Cycle 3: Scheduler switches to Warp 3
...
Cycle 400: Warp 0 becomes ready again
```

This works efficiently because each warp has its own register state, so switching is essentially free compared to a CPU thread switch.

---

## 5. Hardware Constraints: Occupancy and Limits

An SM has hard physical limits.

| Resource | Typical constraint | Effect if exceeded |
| --- | --- | --- |
| Max threads per SM | 2048 | Extra blocks wait in queue |
| Max thread blocks per SM | 32 | Fewer blocks can run concurrently |
| Register file | 65,536 32-bit registers | Lower occupancy |
| Shared memory | Up to 164–228 KB | Fewer blocks can fit |

Occupancy measures how fully the SM is utilized by active warps. High occupancy helps hide memory latency; low occupancy leaves ALUs idle.

## What Is Occupancy?

Occupancy is the fraction of the maximum possible warps that an SM can host at once.

$$
\text{Occupancy} = \frac{\text{Active Warps on the SM}}{\text{Maximum Theoretical Warps Supported by the SM}}
$$

If an SM can support 64 warps but only 32 fit because of register or shared memory pressure, the kernel is running at 50% occupancy.

- High occupancy gives the scheduler more warps to hide memory latency.
- Low occupancy leaves ALUs idle while waiting for memory operations to finish.

## 6. Warp-Level Communication: Shuffle Instructions

Before modern GPUs, threads in a warp that wanted to share data often had to write to shared memory, synchronize, and read the values back.

Modern GPUs support warp shuffle instructions, which let threads exchange register values directly within the warp.

```text
Lane 0 reads from Lane 1 directly
Lane 1 reads from Lane 0 directly
```

This bypasses shared memory and executes very quickly. It is especially useful for reductions such as summing or taking the maximum of values inside a warp.


## Summary

The logical hierarchy is:

$$
\text{Grid} \rightarrow \text{Block} \rightarrow \text{Warp} \rightarrow \text{Thread}
$$

This structure is the foundation of how GPU programs are mapped onto hardware.